In [139]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
class Performer:
    # Shared across ALL Performer instances
    full_history = []  

    def __init__(self, name, role, client_config, model,
                 system_prompt = "", temperature = 0.7, max_tokens = 500):
        """
        A simple performer in the Yes-And game.
        """
        self.name = name
        self.role = role   # "host" or "player"
        self.client_config = client_config  # Store config instead of client
        self.model = model
        self.system_prompt = system_prompt
        self.temperature = temperature
        self.max_tokens = max_tokens

    def _get_client(self):
        """Create client from stored config."""
        if self.client_config.get('base_url'):
            return OpenAI(api_key=self.client_config['api_key'], base_url=self.client_config['base_url'])
        else:
            return OpenAI(api_key=self.client_config.get('api_key'))

    def set_system_prompt(self, system_prompt):
        """Reset or update the system prompt."""
        self.system_prompt = system_prompt

    def _chat(self, extra_user_msg = None):
        """Internal helper to call the model with full history and append response."""
        history = [{"role": "system", "content": self.system_prompt}]
        history.extend(Performer.full_history)
        if extra_user_msg:
            history.append({"role": "user", "content": extra_user_msg})

        client = self._get_client()
        resp = client.chat.completions.create(
            model=self.model,
            messages=history,
            temperature=self.temperature,
            max_tokens=self.max_tokens
        )
        content = resp.choices[0].message.content

        # Label with performer identity (so others know who said it)
        labeled_content = f"{self.name} says: {content}"

        # Save assistant reply into shared history
        Performer.full_history.append({"role": "assistant", "content": labeled_content})
        return labeled_content

    def start_game(self, user_message):
        """First user message to get things started."""
        Performer.full_history.append({"role": "user", "content": user_message})
        return self._chat()

    def user_interaction(self, user_message):
        """Continue conversation with full history (host + user)."""
        Performer.full_history.append({"role": "user", "content": user_message})
        return self._chat()

    def speak(self):
        """
        Player speaks by building on the last assistant message in history.
        Example: Wayne uses Ryan's last line as context for his next move.
        """
        # Find the last assistant message (could be host or another player)
        last_line = None
        for msg in reversed(Performer.full_history):
            if msg["role"] == "assistant":
                last_line = msg["content"]
                break

        if last_line is None:
            # If no assistant messages yet, just let the model go
            return self._chat()

        # Add a user prompt to encourage Yes-And on last line
        extra_prompt = f"Continue the scene by building on this: {last_line}"
        return self._chat(extra_user_msg=extra_prompt)

    def decide(self):
        """
        Host decides if the game should continue or end.
        By default, let the model make the decision based on the history.
        """
        assert self.role == "host", "Only the host should decide."
        decision_prompt = (
            "Based on the scene so far, decide whether to CONTINUE or END GAME. "
            "Reply with only 'continue' or 'end'."
        )
        decision = self._chat(extra_user_msg=decision_prompt)

        # Normalize decision
        if "end" in decision.lower():
            return "end"
        return "continue"

    @classmethod
    def get_full_history(cls):
        """Return the full conversation so far."""
        return cls.full_history

    @classmethod
    def clear_full_history(cls):
        """Reset the shared history."""
        cls.full_history = []


In [142]:
def format_transcript(history) :
    """
    Turn shared history into a single Markdown transcript.
    """
    lines = []
    for msg in history:
        if msg["role"] == "user":
            lines.append(f"**Audience:** {msg['content']}")
        else:
            # assistant lines are already labeled like "Host says: ..."
            lines.append(msg["content"])
    return "\n\n".join(lines)

def run_yes_and_game_stream(host: Performer, p1: Performer, p2: Performer,
                            scenario: str, max_rounds: int = 6):
    """
    Generator that yields the displayed transcript after each action.
    """
    # Fresh game
    Performer.clear_full_history()

    # Kickoff: host greets, user provides scenario
    host.start_game("Welcome to Yes, And! I'll guide the game. What scenario should we play?")
    yield format_transcript(Performer.get_full_history())

    host.user_interaction(f"Scenario: {scenario.strip()}")
    yield format_transcript(Performer.get_full_history())

    # Alternate players with host decisions after each player turn
    players = [p1, p2]
    turn = 0
    for _ in range(max_rounds):
        current = players[turn % 2]
        current.speak()
        yield format_transcript(Performer.get_full_history())

        decision = host.decide()
        yield format_transcript(Performer.get_full_history())

        if decision == "end":
            host.user_interaction(
                "Please give a short 1–2 sentence closing line of wisdom for the players and audience."
            )
            yield format_transcript(Performer.get_full_history())
            break

        turn += 1

In [151]:
# --- Gradio UI ---

def play_game_stream(scenario: str, max_rounds: int, host_state, p1_state, p2_state):
    # unpack performers from gr.State
    host: Performer = host_state
    p1: Performer = p1_state
    p2: Performer = p2_state
    if not scenario.strip():
        yield "⚠️ Please enter a scenario above."
        return
    # stream incremental transcript updates
    for transcript in run_yes_and_game_stream(host, p1, p2, scenario, max_rounds):
        yield transcript

In [ ]:
load_dotenv(override=True)

# create client configurations for different models
anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

# instantiate performers with client configs instead of client objects
host = Performer(name="Drew", role="host", client_config={"api_key": os.getenv('OPENAI_API_KEY')}, model="gpt-4o-mini")
performer1 = Performer(name="Ryan", role="player", client_config={"api_key": os.getenv('GOOGLE_API_KEY'), "base_url": gemini_url}, model="gemini-2.5-flash")
performer2 = Performer(name="Wayne", role="player", client_config={"api_key": os.getenv('ANTHROPIC_API_KEY'), "base_url": anthropic_url}, model="claude-3-5-haiku-latest")

In [146]:

host_system_prompt = f"""You are {host.name}, the host of a multi-agent “Yes, And” improv game.You do not play the game yourself—you only guide it, 
moderate it, and make decisions.
Responsibilities
Solicit Scenario
Ask the user (audience) for a fun scenario to start the game.If unclear/inappropriate, ask them to rephrase once.
Frame the Scene
Convert the scenario into a structured Scene Brief (setting, tone, and constraints).
Broadcast this Scene Brief as instructions to the players. The instructions for {performer1.name} and {performer2.name}
should not be more than 3 sentences. You should not make up names for performers. It is there story to tell. Just 
give them the scene brief and let them play.
Run the Game Loop
Alternate turns between {performer1.name} (Player 1) and {performer2.name} (Player 2).
After each pair of turns, decide whether to continue or end.
If ending, wrap up with a closing message to the user.
Decision Making
Say [HOST DECISION: continue] to keep the game going.
Say [HOST DECISION: End Game] to stop."""

performer1_system_prompt = f"""You are {performer1.name}, Player 1 in the improv game Yes, And.
You play inside the scene brief that {host.name} (the host) provides.
Responsibilities
Always accept what has been established (the “Yes”).Always add something new that pushes the story forward (the “And”).
Stay within the tone, rules, and constraints that {host.name} defines.
Write 2–3 sentences per turn (unless {host.name} specifies otherwise).
Never act as {host.name} or {performer2.name} — only roleplay your own turn."""

performer2_system_prompt = f"""You are {performer2.name}, Player 2 in the improv game Yes, And.
You play inside the scene brief that {host.name} (the host) provides.
Responsibilities
Always accept what has been established (the “Yes”).Always add something new that pushes the story forward (the “And”).
Stay within the tone, rules, and constraints that {host.name} defines.
Write 2–3 sentences per turn (unless {host.name} specifies otherwise).
Never act as {host.name} or {performer1.name} — only roleplay your own turn."""

In [147]:
host.set_system_prompt(host_system_prompt)
performer1.set_system_prompt(performer1_system_prompt)
performer2.set_system_prompt(performer2_system_prompt)


In [ ]:
# Fixed Gradio interface
with gr.Blocks(title="Yes, And — Live Stream Demo") as demo:
    gr.Markdown("# 🎭 Yes, And — Multi-Agent Improv (Live)")
    gr.Markdown("Type your **scenario**, click **Play**, and watch the scene unfold line-by-line.")

    with gr.Row():
        scenario_box = gr.Textbox(
            label="Your Scenario (you decide!)",
            placeholder="e.g., Two rival magicians are stuck in a backstage elevator before curtain.",
            lines=2,
        )
        rounds_slider = gr.Slider(2, 12, value=6, step=1, label="Max Rounds (player turns)")
        play_btn = gr.Button("▶️ Play", variant="primary")

    transcript_md = gr.Markdown("_(transcript will appear here)_")

    # Hold the Python objects in State components (now serializable)
    host_state = gr.State(host)
    p1_state   = gr.State(performer1)
    p2_state   = gr.State(performer2)

    # Hook up the streaming generator: one column, updated after every action
    play_btn.click(
        fn=play_game_stream,
        inputs=[scenario_box, rounds_slider, host_state, p1_state, p2_state],
        outputs=transcript_md,
        show_progress="full",
        queue=True,            # allows streaming yields
    )
    demo.launch()


In [152]:


with gr.Blocks(title="Yes, And — Live Stream Demo") as demo:
    gr.Markdown("# 🎭 Yes, And — Multi-Agent Improv (Live)")
    gr.Markdown("Type your **scenario**, click **Play**, and watch the scene unfold line-by-line.")

    with gr.Row():
        scenario_box = gr.Textbox(
            label="Your Scenario (you decide!)",
            placeholder="e.g., Two rival magicians are stuck in a backstage elevator before curtain.",
            lines=2,
        )
        rounds_slider = gr.Slider(2, 12, value=6, step=1, label="Max Rounds (player turns)")
        play_btn = gr.Button("▶️ Play", variant="primary")

    transcript_md = gr.Markdown("_(transcript will appear here)_")

     # Hold the Python objects in State components
    host_state = gr.State(host)
    p1_state   = gr.State(performer1)
    p2_state   = gr.State(performer2)

    # Hook up the streaming generator: one column, updated after every action
    play_btn.click(
        fn=play_game_stream,
        inputs=[scenario_box, rounds_slider, host_state, p1_state, p2_state],
        outputs=transcript_md,
        show_progress="full",
        queue=True,            # allows streaming yields
    )
    demo.launch()

TypeError: The initial value of `gr.State` must be able to be deepcopied. The initial value of type <class '__main__.Performer'> cannot be deepcopied.

In [135]:
print(Performer.get_full_history())

[]
